In [ ]:
# ==============================================================================
#  (ZINDI FINANCIAL HEALTH)
# Includes Data Cleaning, Unsupervised FE, Drift Removal, SMOTE, & F1 Optimizer
# ==============================================================================

# ---------------------------------------------------------
# 1. SETUP & INSTALLATION
# ---------------------------------------------------------
!pip install xgboost lightgbm imbalanced-learn scikit-learn scipy --quiet

import pandas as pd
import numpy as np
import warnings
import os
import scipy as sp
from functools import partial
from google.colab import files

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 2. UPLOAD & LOAD DATA FROM LOCAL COMPUTER
# ---------------------------------------------------------
print(">>> [STEP 1] Upload Data")
print("Please click 'Choose Files' below and upload: Train.csv, Test.csv, and SampleSubmission.csv")

# This will open the file picker dialogue in Google Colab
uploaded = files.upload()

print("\n>>> Reading uploaded files...")
try:
    train = pd.read_csv('Train.csv')
    test = pd.read_csv('Test.csv')
    sample_sub = pd.read_csv('SampleSubmission.csv')
except FileNotFoundError:
    print("❌ ERROR: CSV files not found. Please ensure you uploaded all three files!")
    raise

# Merge Data for consistent preprocessing
train['is_train'] = 1
test['is_train'] = 0
test['Target'] = np.nan
data = pd.concat([train, test], ignore_index=True)

# ---------------------------------------------------------
# 3. DATA CLEANING MODULE (THE JANITOR)
# ---------------------------------------------------------
print("\n>>> [STEP 2] Cleaning Messy Text & Outliers...")

cat_cols = data.select_dtypes(include=['object']).columns
for col in cat_cols:
    if col not in ['ID', 'Target']:
        # Standardize strings: lowercase and strip extra spaces
        data[col] = data[col].astype(str).str.lower().str.strip()
        # Unify missing value representations
        data[col] = data[col].replace({'nan': np.nan, 'none': np.nan, 'null': np.nan, '': np.nan, 'n/a': np.nan})

# Handle extreme missing values (>40% missing)
for col in cat_cols:
    if col not in ['ID', 'Target'] and data[col].isnull().mean() > 0.4:
        data[col] = data[col].fillna("unknown_missing")

# Winsorization: Clip extreme numeric outliers (1st to 99th percentile)
num_cols = data.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if col not in ['Target', 'is_train']:
        data[col] = data[col].clip(data[col].quantile(0.01), data[col].quantile(0.99))

# ---------------------------------------------------------
# 4. FEATURE ENGINEERING (COUNTRY SCALING)
# ---------------------------------------------------------
print(">>> [STEP 3] Calculating Economic Features (USD, Z-Score, Percentiles)...")

exchange_rates = {
    'zimbabwe': 1.0, 'malawi': 0.0012, 'tanzania': 0.00043,
    'zambia': 0.053, 'mozambique': 0.016, 'uganda': 0.00027
}
money_cols = ['personal_income', 'business_expenses', 'business_turnover']

for col in money_cols:
    if data[col].dtype == object:
         data[col] = data[col].astype(str).str.replace(',', '').astype(float)

    rate = data['country'].map(exchange_rates).fillna(1.0)
    data[f'{col}_usd'] = data[col] * rate

    # Impute missing with country-specific median
    median_usd = data.groupby('country')[f'{col}_usd'].transform('median')
    data[f'{col}_usd'] = data[f'{col}_usd'].fillna(median_usd).fillna(0)

    # Log Transform
    data[f'log_{col}_usd'] = np.log1p(data[f'{col}_usd'])

    # Country-Specific Z-Score & Percentile Ranking
    data[f'{col}_country_zscore'] = data.groupby('country')[f'log_{col}_usd'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
    data[f'{col}_country_pct'] = data.groupby('country')[f'log_{col}_usd'].rank(pct=True)

ordinal_map = {
    'never had': 0, "don't know": 0, "don't know / doesn't apply": 0, "don’t know or n/a": 0, 'no': 0,
    'used to have but don’t have now': 1, "used to have but don't have now": 1, 'yes, sometimes': 1,
    'have now': 2, 'yes': 2, 'yes, always': 2
}
ordinal_cols = ['motor_vehicle_insurance', 'has_mobile_money', 'has_internet_banking', 'has_credit_card', 'has_insurance', 'has_loan_account', 'keeps_financial_records', 'medical_insurance', 'funeral_insurance', 'has_debit_card']

for col in ordinal_cols:
    if col in data.columns:
        data[f'{col}_ord'] = data[col].map(ordinal_map).fillna(0).astype(int)

# ---------------------------------------------------------
# 5. UNSUPERVISED FEATURES (K-MEANS & PCA)
# ---------------------------------------------------------
print(">>> [STEP 4] Extracting SME Archetypes (PCA & K-Means)...")

unsup_cols = [f'log_{c}_usd' for c in money_cols] + [f'{c}_ord' for c in ordinal_cols if f'{c}_ord' in data.columns]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[unsup_cols].fillna(0))

# PCA for dimensionality reduction
pca = PCA(n_components=3, random_state=42)
pca_features = pca.fit_transform(X_scaled)
data['pca_eco_1'] = pca_features[:, 0]
data['pca_eco_2'] = pca_features[:, 1]
data['pca_eco_3'] = pca_features[:, 2]

# K-Means clustering for behavioral grouping
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
data['sme_cluster'] = kmeans.fit_predict(X_scaled)

# ---------------------------------------------------------
# 6. ADVERSARIAL DRIFT REMOVAL
# ---------------------------------------------------------
print("\n>>> [STEP 5] Adversarial Validation (Removing Biased Features)...")

audit_data = data.copy()
audit_data['is_test'] = (audit_data['is_train'] == 0).astype(int)
drop_audit = ['ID', 'Target', 'is_train', 'is_test'] + money_cols + [f'{c}_usd' for c in money_cols]
features_check = [c for c in audit_data.columns if c not in drop_audit]

for col in audit_data[features_check].select_dtypes(include='object').columns:
    le = LabelEncoder()
    audit_data[col] = le.fit_transform(audit_data[col].astype(str))

dtrain_audit = lgb.Dataset(audit_data[features_check], label=audit_data['is_test'])
params_audit = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'seed': 42}
model_audit = lgb.train(params_audit, dtrain_audit, num_boost_round=100)

imp = pd.DataFrame({'feature': features_check, 'gain': model_audit.feature_importance(importance_type='gain')}).sort_values('gain', ascending=False)
drifting_feature = imp.iloc[0]['feature']
print(f"   [-] Feature '{drifting_feature}' removed due to train/test distribution drift.")

# ---------------------------------------------------------
# 7. FINAL DATA PREPARATION
# ---------------------------------------------------------
train_df = data[data['is_train'] == 1].copy()
test_df = data[data['is_train'] == 0].copy()

le_target = LabelEncoder()
y = le_target.fit_transform(train_df['Target'])

final_drop = ['ID', 'Target', 'is_train'] + money_cols + [f'{c}_usd' for c in money_cols] + [drifting_feature]
X = train_df.drop(final_drop, axis=1)
X_test = test_df.drop(final_drop, axis=1)

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# Safe Missing Value Imputation
X[cat_cols] = X[cat_cols].fillna("Missing")
X[num_cols] = X[num_cols].fillna(-999)
X_test[cat_cols] = X_test[cat_cols].fillna("Missing")
X_test[num_cols] = X_test[num_cols].fillna(-999)

# Full Label Encoding required for HistGradient and SMOTE
for col in cat_cols:
    l = LabelEncoder()
    full = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    l.fit(full)
    X[col] = l.transform(X[col].astype(str))
    X_test[col] = l.transform(X_test[col].astype(str))

X = X.astype(float)
X_test = X_test.astype(float)

# ---------------------------------------------------------
# 8. TRAINING: SMOTE + ISOTONIC CALIBRATION & OOF TRACKING
# ---------------------------------------------------------
print("\n>>> [STEP 6] Training Ensemble & Generating OOF Predictions...")

seeds = [42, 2023, 777]
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

test_probs = np.zeros((len(X_test), 3))
oof_probs = np.zeros((len(X), 3)) # To track Out-Of-Fold predictions for optimizer

for seed in seeds:
    p_xgb = {'n_estimators': 1500, 'learning_rate': 0.03, 'max_depth': 6, 'objective': 'multi:softprob', 'num_class': 3, 'tree_method': 'hist', 'random_state': seed, 'verbosity': 0}
    p_lgb = {'n_estimators': 1500, 'learning_rate': 0.03, 'max_depth': 8, 'objective': 'multiclass', 'num_class': 3, 'random_state': seed, 'verbosity': -1, 'class_weight': 'balanced'}
    p_hist = {'max_iter': 1500, 'learning_rate': 0.03, 'max_depth': 7, 'random_state': seed}

    seed_test_preds = np.zeros((len(X_test), 3))
    seed_oof_preds = np.zeros((len(X), 3))

    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]

        # 1. SMOTE (Balancing training data only)
        smote = SMOTE(random_state=seed)
        X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)

        # 2. XGBoost + Isotonic Calibration
        m_xgb = xgb.XGBClassifier(**p_xgb)
        m_xgb.fit(X_tr_res, y_tr_res, eval_set=[(X_va, y_va)], verbose=False)
        cal_xgb = CalibratedClassifierCV(estimator=m_xgb, method='isotonic', cv='prefit')
        cal_xgb.fit(X_va, y_va)
        prob_xgb_test = cal_xgb.predict_proba(X_test)
        prob_xgb_val = cal_xgb.predict_proba(X_va)

        # 3. LightGBM + Isotonic Calibration
        m_lgb = lgb.LGBMClassifier(**p_lgb)
        m_lgb.fit(X_tr_res, y_tr_res, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
        cal_lgb = CalibratedClassifierCV(estimator=m_lgb, method='isotonic', cv='prefit')
        cal_lgb.fit(X_va, y_va)
        prob_lgb_test = cal_lgb.predict_proba(X_test)
        prob_lgb_val = cal_lgb.predict_proba(X_va)

        # 4. HistGradientBoosting + Isotonic Calibration
        m_hist = HistGradientBoostingClassifier(**p_hist)
        m_hist.fit(X_tr_res, y_tr_res)
        cal_hist = CalibratedClassifierCV(estimator=m_hist, method='isotonic', cv='prefit')
        cal_hist.fit(X_va, y_va)
        prob_hist_test = cal_hist.predict_proba(X_test)
        prob_hist_val = cal_hist.predict_proba(X_va)

        # Blend probabilities for the test set
        seed_test_preds += (prob_xgb_test + prob_lgb_test + prob_hist_test) / 30

        # Accumulate Validation (OOF) predictions
        seed_oof_preds[val_idx] += (prob_xgb_val + prob_lgb_val + prob_hist_val) / 3

    test_probs += seed_test_preds / len(seeds)
    oof_probs += seed_oof_preds / len(seeds)
    print(f"   ✅ Seed {seed} Finished.")

# ---------------------------------------------------------
# 9. F1-MACRO OPTIMIZER & GENERATE SUBMISSION
# ---------------------------------------------------------
print("\n>>> [STEP 7] Running Scipy Optimizer for Max F1-Macro...")

class F1Optimizer():
    def __init__(self):
        pass

    def __call__(self, coef, X, y):
        x_p = np.copy(X)
        for i in range(3):
            x_p[:, i] = x_p[:, i] * coef[i]

        preds = np.argmax(x_p, axis=1)
        # Return negative F1 because Scipy minimizes the function
        return -f1_score(y, preds, average='macro')

    def fit(self, X, y):
        loss_partial = partial(self.__call__, X=X, y=y)
        initial_coef = [1.0, 1.0, 1.0] # Standard Argmax weights
        self.coef_ = sp.optimize.minimize(loss_partial, initial_coef, method='nelder-mead')

print("   Finding optimal threshold weights on Validation Data...")
opt = F1Optimizer()
opt.fit(oof_probs, y)
best_weights = opt.coef_['x']

print(f"   Standard Weights (Argmax) : [1.0, 1.0, 1.0]")
print(f"   Optimal Weights (F1-Macro): {best_weights}")

# Apply optimized weights to the Test Data probabilities
for i in range(3):
    test_probs[:, i] = test_probs[:, i] * best_weights[i]

# Determine final predictions using the calibrated & weighted probabilities
final_preds_idx = np.argmax(test_probs, axis=1)
final_labels = le_target.inverse_transform(final_preds_idx)

# Prepare submission dataframe
submission = pd.DataFrame({'ID': test_df['ID'], 'Target': final_labels})
submission = pd.merge(sample_sub[['ID']], submission, on='ID', how='left')

# --- NAMA SEDERHANA ---
filename = 'submission_final.csv'
submission.to_csv(filename, index=False)

print(f"\n>>> DONE! The file {filename} is ready and will be downloaded now.")
files.download(filename)


>>> [STEP 1] Upload Data
Please click 'Choose Files' below and upload: Train.csv, Test.csv, and SampleSubmission.csv


Saving SampleSubmission.csv to SampleSubmission.csv
Saving Test.csv to Test.csv
Saving Train.csv to Train.csv

>>> Reading uploaded files...

>>> [STEP 2] Cleaning Messy Text & Outliers...
>>> [STEP 3] Calculating Economic Features (USD, Z-Score, Percentiles)...
>>> [STEP 4] Extracting SME Archetypes (PCA & K-Means)...

>>> [STEP 5] Adversarial Validation (Removing Biased Features)...
   [-] Feature 'pca_eco_3' removed due to train/test distribution drift.

>>> [STEP 6] Training Ensemble & Generating OOF Predictions...
   ✅ Seed 42 Finished.
   ✅ Seed 2023 Finished.
   ✅ Seed 777 Finished.

>>> [STEP 7] Running Scipy Optimizer for Max F1-Macro...
   Finding optimal threshold weights on Validation Data...
   Standard Weights (Argmax) : [1.0, 1.0, 1.0]
   Optimal Weights (F1-Macro): [0.94784594 0.94139446 1.06594436]

>>> DONE! The file submission_final.csv is ready and will be downloaded now.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>